In [ ]:
from collections import defaultdict
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

DATA_ROOT = Path("./50Cities")
CROMA_PY_DIR = Path(".")
PRETRAINED_PATH = Path("./checkpoints/croma_base.pt")
OUTPUT_DIR = Path("./runs/50cities_croma_base")
EPOCHS = 30
BATCH_SIZE = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0
SEED = 42
PATCH_SIZE = 128
SPATIAL_BLOCK_SIZE = 512
NUM_CLASSES = 13
IGNORE_INDEX = 255
CLASS_WEIGHTS = (1.0,) * NUM_CLASSES


LABEL_FILENAME = "label.png"
BACKGROUND_VALUE = IGNORE_INDEX

S2_INPUT_BANDS = (
    "B01", "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B09", "B11", "B12",
)
CLASS_NAMES = (
    "Low Vegetation Bare", "Low Vegetation Non-Bare",
    "High Vegetation Bare", "High Vegetation Non-Bare",
    "Bare Soil", "Sand", "Rock", "Snow", "Water", "Industry",
    "Urban Low Density", "Urban Medium Density", "Urban High Density",
)

CLASS_COLORS = (
    (150, 184, 16),
    (238, 255, 0),
    (126, 236, 219),
    (0, 174, 125),
    (255, 192, 0),
    (234, 114, 7),
    (88, 68, 68),
    (134, 217, 255),
    (0, 0, 125),
    (132, 132, 132),
    (255, 255, 255),
    (253, 0, 181),
    (255, 0, 0),
)
COLOR_TO_CLASS = {
    color: class_id
    for class_id, color in enumerate(CLASS_COLORS)
}

COLOR_TO_CLASS[(118, 118, 118)] = COLOR_TO_CLASS[(132, 132, 132)]
assert len(CLASS_COLORS) == NUM_CLASSES
assert len(set(CLASS_COLORS)) == NUM_CLASSES


CONFIRM_S2_BAND_ORDER = True
S1_BAND_INDICES = (1, 2)

CROMA_BANDS = (
    "B01", "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B09", "B11", "B12",
)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)


In [ ]:
## Dataset

def normalize_channels(image, invalid=None):
    """Normalize valid pixels per channel using clipped mean and variance."""
    image = image.astype(np.float32, copy=False)
    output = np.zeros_like(image, dtype=np.float32)
    for index, channel in enumerate(image):
        valid = np.isfinite(channel)
        if invalid is not None:
            valid &= ~invalid[index]
        if not valid.any():
            continue
        values = channel[valid]
        mean, std = values.mean(), values.std()
        if std < 1e-6:
            continue
        output[index, valid] = np.clip(
            (values - (mean - 2 * std)) / (4 * std), 0, 1
        )
    return output


def read_window(path, row, col, size, indices=None):
    """Read a geospatial tile without resampling."""
    from rasterio.windows import Window

    with rasterio.open(path) as src:
        array = src.read(indexes=indices, window=Window(col, row, size, size))
        if array.ndim == 2:
            array = array[None]
        if array.shape[-2:] != (size, size):
            raise ValueError(f"Incomplete tile in {path}: {array.shape}")
        nodata = src.nodata
    invalid = ~np.isfinite(array)
    if nodata is not None and np.isfinite(nodata):
        invalid |= array == nodata
    return array, invalid


def expand_s2(image):
    """Map declared Sentinel-2 bands into CROMA's 12-band order."""
    expanded = np.zeros((12, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
    for index, band in enumerate(S2_INPUT_BANDS):
        expanded[CROMA_BANDS.index(band)] = image[index]
    return expanded


def find_labeled_cities():
    """Select scenes with matching pixel dimensions and PNG annotations."""
    from PIL import Image

    cities = []
    for city in city_dirs:
        paths = [city / name for name in ("S1.tif", "S2.tif", LABEL_FILENAME)]
        if not all(path.is_file() for path in paths):
            continue
        with rasterio.open(paths[0]) as s1, rasterio.open(paths[1]) as s2:
            with Image.open(paths[2]) as label:
                label_shape = (label.height, label.width)
            if s1.shape != s2.shape or s1.shape != label_shape:
                raise ValueError(f"Unaligned image dimensions in {city}")
            if s1.transform != s2.transform or s1.crs != s2.crs:
                print(
                    f"Warning: {city.name} has different S1/S2 georeferencing; "
                    "using original pixel coordinates without resampling."
                )
            if s1.count != 2 or s2.count != len(S2_INPUT_BANDS):
                raise ValueError(f"Unexpected sensor band counts in {city}")
        cities.append(city)
    return cities


def read_label_window(path, row, col, size):
    """Read an RGBA mask crop and map opaque known colors to class IDs."""
    from PIL import Image

    with Image.open(path) as image:
        rgba = np.asarray(
            image.convert("RGBA").crop((col, row, col + size, row + size))
        )
    target = np.full((size, size), IGNORE_INDEX, dtype=np.int64)
    opaque = rgba[:, :, 3] == 255
    for color, class_id in COLOR_TO_CLASS.items():
        matched = np.all(rgba[:, :, :3] == color, axis=-1)
        target[opaque & matched] = class_id
    return target


def audit_label_palette(cities):
    """Audit opaque labels and ignore only negligible unmapped colors."""
    from collections import Counter
    from PIL import Image

    unknown = Counter()
    total_opaque = 0
    for city in tqdm(cities, desc="Auditing PNG labels"):
        with Image.open(city / LABEL_FILENAME) as image:
            rgba = np.asarray(image.convert("RGBA"))
        opaque_rgb = rgba[rgba[:, :, 3] == 255, :3]
        total_opaque += len(opaque_rgb)
        colors, counts = np.unique(
            opaque_rgb, axis=0, return_counts=True
        )
        for color, count in zip(colors, counts):
            key = tuple(int(value) for value in color)
            if key not in COLOR_TO_CLASS:
                unknown[key] += int(count)

    unknown_pixels = sum(unknown.values())
    unknown_ratio = (
        unknown_pixels / total_opaque if total_opaque else 0.0
    )
    print(f"Total opaque pixels: {total_opaque:,}")
    print(f"Unmapped pixels: {unknown_pixels:,}")
    print(f"Unmapped ratio: {unknown_ratio:.6%}")
    print("Unmapped opaque colors:", unknown.most_common(30))
    if unknown_ratio > 0.0001:
        raise ValueError(
            "Too many unknown label colors: verify the RGB mapping."
        )
    if unknown_pixels:
        print("Rare unmapped colors will use IGNORE_INDEX.")


def build_spatial_splits(cities):
    """Assign spatial blocks per city to disjoint dataset splits."""
    rng = np.random.default_rng(SEED)
    samples = defaultdict(list)
    for city in cities:
        with rasterio.open(city / "S1.tif") as src:
            height, width = src.shape
        blocks = []
        for block_row in range(0, height, SPATIAL_BLOCK_SIZE):
            for block_col in range(0, width, SPATIAL_BLOCK_SIZE):
                tiles = []
                for row in range(block_row, min(block_row + SPATIAL_BLOCK_SIZE,
                                                height - PATCH_SIZE + 1), PATCH_SIZE):
                    for col in range(block_col, min(block_col + SPATIAL_BLOCK_SIZE,
                                                    width - PATCH_SIZE + 1), PATCH_SIZE):
                        if row + PATCH_SIZE <= height and col + PATCH_SIZE <= width:
                            tiles.append((city, row, col))
                if tiles:
                    blocks.append(tiles)
        rng.shuffle(blocks)
        if len(blocks) < 3:
            raise ValueError(f"Too few spatial blocks in {city}")
        n_test = max(1, round(len(blocks) * 0.20))
        n_val = max(1, round(len(blocks) * 0.16))
        for split, subset in (
            ("test", blocks[:n_test]),
            ("val", blocks[n_test:n_test + n_val]),
            ("train", blocks[n_test + n_val:]),
        ):
            for block in subset:
                samples[split].extend(block)
    return samples


class FiftyCitiesDataset(Dataset):
    """Load paired Sentinel-1/Sentinel-2 tiles and mapped land-cover labels."""

    def __init__(self, samples):
        self.samples = list(samples)
        if not self.samples:
            raise ValueError("No tiles in this split.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        city, row, col = self.samples[index]
        sar, sar_invalid = read_window(
            city / "S1.tif", row, col, PATCH_SIZE, S1_BAND_INDICES
        )
        optical, optical_invalid = read_window(
            city / "S2.tif", row, col, PATCH_SIZE
        )
        target = read_label_window(
            city / LABEL_FILENAME, row, col, PATCH_SIZE
        )
        invalid = sar_invalid.any(axis=0) | optical_invalid.any(axis=0)
        target[invalid] = IGNORE_INDEX
        sar = normalize_channels(sar, sar_invalid)
        optical = expand_s2(normalize_channels(optical, optical_invalid))
        return (
            torch.from_numpy(sar.copy()),
            torch.from_numpy(optical.copy()),
            torch.from_numpy(target.copy()),
        )


assert len(CLASS_NAMES) == len(CLASS_COLORS) == NUM_CLASSES
assert len(set(CLASS_COLORS)) == NUM_CLASSES
assert len(S2_INPUT_BANDS) == 12
assert set(S2_INPUT_BANDS) == set(CROMA_BANDS)
if not CONFIRM_S2_BAND_ORDER:
    raise RuntimeError(
        "Check the actual Sentinel-2 GeoTIFF band order, then set "
        "CONFIRM_S2_BAND_ORDER = True in the configuration cell."
    )
labeled_cities = find_labeled_cities()
print("Labeled cities:", len(labeled_cities), [c.name for c in labeled_cities])
audit_label_palette(labeled_cities)
splits = build_spatial_splits(labeled_cities)
train_dataset = FiftyCitiesDataset(splits["train"])
val_dataset = FiftyCitiesDataset(splits["val"])
test_dataset = FiftyCitiesDataset(splits["test"])
for name in ("train", "val", "test"):
    print(name, len(splits[name]))
sar, optical, labels = train_dataset[0]
print("Example shapes:", sar.shape, optical.shape, labels.shape)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")


In [ ]:
## Model

sys.path.insert(0, str(CROMA_PY_DIR.resolve()))
from use_croma import PretrainedCROMA # from https://github.com/antofuller/CROMA


class CROMASegmenter(nn.Module):
    """Full-fine-tuning model using the original CROMA-base encoder."""

    def __init__(self, pretrained_path):
        super().__init__()
        self.backbone = PretrainedCROMA(
            pretrained_path=str(pretrained_path),
            size="base",
            modality="both",
            image_resolution=128,
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(768 * 3, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(128, NUM_CLASSES, kernel_size=1),
        )

    def forward(self, sar, optical):
        outputs = self.backbone(SAR_images=sar, optical_images=optical)
        spatial_maps = []
        for key in ("SAR_encodings", "optical_encodings", "joint_encodings"):
            tokens = outputs[key]
            spatial_maps.append(
                tokens.transpose(1, 2).reshape(sar.shape[0], 768, 16, 16)
            )
        logits = self.decoder(torch.cat(spatial_maps, dim=1))
        return F.interpolate(
            logits, size=(128, 128), mode="bilinear", align_corners=False
        )


model = CROMASegmenter(PRETRAINED_PATH).to(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
assert all(parameter.requires_grad for parameter in model.backbone.parameters())

with torch.no_grad():
    model.eval()
    sample_logits = model(sar[None].to(DEVICE), optical[None].to(DEVICE))
print("Output:", sample_logits.shape)
assert sample_logits.shape == (1, NUM_CLASSES, 128, 128)

In [ ]:
## Loss and metrics

class_weights = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)


def segmentation_loss(logits, target):
    """Compute weighted cross-entropy plus masked soft Dice."""
    valid = target != IGNORE_INDEX
    if not valid.any():
        return logits.sum() * 0.0
    ce = F.cross_entropy(
        logits, target, weight=class_weights, ignore_index=IGNORE_INDEX
    )
    safe_target = target.masked_fill(~valid, 0)
    one_hot = F.one_hot(safe_target, num_classes=NUM_CLASSES)
    one_hot = one_hot.permute(0, 3, 1, 2).float()
    mask = valid.unsqueeze(1)
    probabilities = logits.softmax(dim=1) * mask
    one_hot = one_hot * mask
    intersection = (probabilities * one_hot).sum(dim=(0, 2, 3))
    denominator = (probabilities + one_hot).sum(dim=(0, 2, 3))
    dice = 1.0 - ((2.0 * intersection + 1e-6) / (denominator + 1e-6)).mean()
    return ce + dice


def metrics_from_confusion(confusion):
    """Calculate per-class IoU, F1 and macro IoU."""
    matrix = confusion.float()
    true_positive = matrix.diag()
    false_positive = matrix.sum(dim=0) - true_positive
    false_negative = matrix.sum(dim=1) - true_positive
    union = true_positive + false_positive + false_negative
    iou = torch.where(union > 0, true_positive / union.clamp(min=1), float("nan"))
    f1_denom = 2 * true_positive + false_positive + false_negative
    f1 = torch.where(
        f1_denom > 0,
        2 * true_positive / f1_denom.clamp(min=1),
        float("nan"),
    )
    return {
        "miou": torch.nanmean(iou).item(),
        "iou": iou.tolist(),
        "f1": f1.tolist(),
        "support": matrix.sum(dim=1).int().tolist(),
    }


def run_epoch(model, loader, optimizer=None):
    """Train or evaluate one epoch and accumulate confusion counts."""
    training = optimizer is not None
    model.train(training)
    confusion = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long, device=DEVICE)
    total_loss = 0.0
    total_images = 0
    for sar_batch, optical_batch, target in tqdm(loader, leave=False):
        sar_batch = sar_batch.to(DEVICE, non_blocking=True)
        optical_batch = optical_batch.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training):
            logits = model(sar_batch, optical_batch)
            loss = segmentation_loss(logits, target)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        predicted = logits.argmax(dim=1)
        valid = target != IGNORE_INDEX
        counts = torch.bincount(
            (target[valid] * NUM_CLASSES + predicted[valid]).reshape(-1), minlength=NUM_CLASSES ** 2
        )
        confusion += counts.reshape(NUM_CLASSES, NUM_CLASSES)
        total_loss += loss.item() * target.shape[0]
        total_images += target.shape[0]
    result = metrics_from_confusion(confusion.cpu())
    result["loss"] = total_loss / max(total_images, 1)
    return result

In [ ]:
## Full fine-tuning and best validation checkpoint

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
history = []
best_miou = -1.0
best_path = OUTPUT_DIR / "best_croma_base_segmentation.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader)
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, "
        f"mIoU={train_metrics['miou']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, "
        f"mIoU={val_metrics['miou']:.4f}"
    )
    if val_metrics["miou"] > best_miou:
        best_miou = val_metrics["miou"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_metrics": val_metrics,
                "s2_input_bands": S2_INPUT_BANDS,
                "class_names": CLASS_NAMES,
                "class_colors": CLASS_COLORS,
                "background_value": BACKGROUND_VALUE,
                "ignore_index": IGNORE_INDEX,
            },
            best_path,
        )
        print("Saved best model:", best_path)

In [ ]:
## Learning curves

epochs = [entry["epoch"] for entry in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split in ("train", "val"):
    axes[0].plot(epochs, [entry[split]["loss"] for entry in history], label=split)
    axes[1].plot(epochs, [entry[split]["miou"] for entry in history], label=split)
axes[0].set_title("Loss")
axes[1].set_title("mIoU")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
## Evaluate on the held-out test blocks and per city

checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()
print("Global test metrics:", run_epoch(model, test_loader))
for city in labeled_cities:
    city_samples = [sample for sample in splits["test"] if sample[0] == city]
    if not city_samples:
        continue
    city_loader = DataLoader(
        FiftyCitiesDataset(city_samples), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS,
    )
    print(city.name, run_epoch(model, city_loader))


In [ ]:
## Visualize predictions

sar, optical, target = test_dataset[1]
with torch.no_grad():
    logits = model(sar[None].to(DEVICE), optical[None].to(DEVICE))
prediction = logits.argmax(dim=1)[0].cpu().numpy()
rgb = optical[[3, 2, 1]].permute(1, 2, 0).numpy()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(rgb)
axes[0].set_title("Sentinel-2 RGB")
axes[1].imshow(np.ma.masked_where(target.numpy() == IGNORE_INDEX,
                                 target.numpy()), vmin=0, vmax=12, cmap="tab20")
axes[1].set_title("Ground truth")
axes[2].imshow(prediction, vmin=0, vmax=12, cmap="tab20")
axes[2].set_title("CROMA prediction")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()
